# Rowan's model, first iteration (MOT17)

One network that finds people and links them from frame to frame:

1. **CNN:** 9 convolution layers (ReLU) and 3 max-pooling layers turn a pre-processed frame (3 × 1088 × 1920) into a 64 × 17 × 30 feature map.
2. **Fully connected layers (sigmoid):** FC1–FC3 predict, for each cell of a 17 × 30 grid, *person here?* and the box `cx, cy, w, h`.
3. **Transformer:** each kept cell's 5 numbers become a 64-number token; 3 transformer encoder layers process a frame's tokens.
4. **Comparison with the previous frame:** each token of frame t is scored against every transformer output of frame t−1 and a learned "new person" vector (dot product ÷ √64, softmax). The most likely option is its identity.

Trained with cross-entropy only (person + box + ID), compared with the DeepSORT baseline on MOT17-13 (test).

* **Input:** exactly the data `Exploratory data analysis.ipynb` prepared, through `mot17_data.py` (frame pairs, already resized to 1920 × 1088 and normalized). Run that notebook first.
* **Read-only:** `train/`, `test/` and `gt.txt` are never changed. Everything is written to `outputs/model/`.
* **GPU required.** Run with **Restart kernel → Run all**. Finished training runs are skipped on a re-run (their history is in `outputs/model/history/`).

## M.0 Setup
GPU, seeds and every setting in one place.

In [ ]:
# ---- Imports ----
import inspect
import itertools
import json
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import DataLoader, ConcatDataset

# The shared data code written by 'Exploratory data analysis.ipynb' (never edited here).
import mot17_data
from mot17_data import (DATA_ROOT, DETECTOR_COPY, read_prepared, boxes_to_pixels, batch_to_device,
                        MOT17Frames, MOT17FramePairs, collate_frames, collate_pairs)

DATA_VERSION = mot17_data.data_version()      # stops if the prepared data is missing or changed

# ---- GPU (required) ----
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. This notebook needs the GPU: install the CUDA build of PyTorch "
                       "(see requirements.txt) with no notebook kernel running, then restart.")
DEVICE = torch.device("cuda")
GPU_NAME = torch.cuda.get_device_name(0)
torch.backends.cudnn.benchmark = False        # don't let cuDNN choose algorithms by timing them
torch.backends.cudnn.deterministic = True     # deterministic cuDNN algorithms, so runs are repeatable

# ---- Output folders ----
OUT = mot17_data.PROJECT_ROOT / "outputs" / "model"
CHECKPOINTS, HISTORY, FIGURES, TRACKERS, EVAL = (OUT / d for d in ["checkpoints", "history", "figures", "trackers", "eval"])
for folder in [CHECKPOINTS, HISTORY, FIGURES, FIGURES / "runs", TRACKERS, EVAL]:
    folder.mkdir(parents=True, exist_ok=True)
LOG_FILE = OUT / "progress.log"               # one line per epoch, so a long run can be followed from outside


def log(message):
    """Print a line and append it to progress.log."""
    line = f"{time.strftime('%Y-%m-%d %H:%M:%S')}  {message}"
    print(line)
    with open(LOG_FILE, "a", encoding="utf-8") as f:
        f.write(line + "\n")


print("GPU:", GPU_NAME, "| torch", torch.__version__, "| data version", DATA_VERSION)

In [ ]:
# ---- Settings (every value used below) ----
BATCH_SIZE = 64            # pairs per Adam step (Rowan's starting value; searched in M.9)
LEARNING_RATE = 0.001      # Rowan's starting value; searched in M.9
EPOCHS = 16                # Rowan's starting value; searched in M.9
MAX_EPOCHS = 24            # every run trains this long; epochs 8, 16 and 24 are read from its checkpoints
SAVE_EPOCHS = [8, 16, 24]
# Optimizer: Adam with default betas and no weight decay.
MICRO_BATCH = 4            # pairs on the GPU at once; gradients are added up until BATCH_SIZE pairs (M.3)
GRID_ROWS, GRID_COLS = 17, 30          # the prediction grid: one cell = 64 x 64 pixels of the 1920 x 1088 frame
CELL_OUTPUTS = 5                       # person here?, cx, cy, w, h
TOKEN_SIZE = 64
PERSON_THRESHOLD = 0.5     # a cell is "a person" when person here? > 0.5 (validation, test, tracking)
NUM_WORKERS = 8            # processes loading frames in parallel
SEED = 0

# M.9 grid search (validation only)
GRID_BATCH_SIZES = [32, 64, 128]
GRID_LEARNING_RATES = [0.0001, 0.001, 0.01]
GRID_EPOCHS = [8, 16, 24]
GRID_OPTION_SMALL = False  # OPTION: only batch 32, 64 x learning rate 0.0001, 0.001 (4 runs), if time is short
ALLOW_LONG_GRID = False    # set True only after Rowan agrees to a grid estimated at more than MAX_GRID_HOURS
MAX_GRID_HOURS = 10

SAMPLE_VIDEO, SAMPLE_FRAME = "MOT17-13", 300   # the M.8 sample image

# Chart colours (one per split, used in every curve)
SPLIT_COLOURS = {"train": "#2a78d6", "val": "#eb6834", "test": "#1baf7a"}
SPLIT_NAMES = {"train": "training", "val": "validation", "test": "testing"}

assert BATCH_SIZE % MICRO_BATCH == 0 and all(b % MICRO_BATCH == 0 for b in GRID_BATCH_SIZES)
assert (mot17_data.TARGET_HEIGHT, mot17_data.TARGET_WIDTH) == (GRID_ROWS * 64, GRID_COLS * 64)


def seed_everything(seed=SEED):
    """Same random numbers every run (weight initialisation and the order of the training pairs)."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything()

# MD5 of the raw annotation files, checked again at the end (M.11) to prove nothing changed.
VIDEOS = read_prepared("videos")
RAW_FILES = [DATA_ROOT / folder / sub for folder in VIDEOS["folder"] for sub in ["gt/gt.txt", "seqinfo.ini"]]
raw_md5_before = {str(p): mot17_data.file_md5(p) for p in RAW_FILES}

## M.1 Training targets (from the cleaned boxes; nothing is altered)
Each box goes to the grid cell that holds its centre: row = ⌊`cy` × 17⌋, column = ⌊`cx` × 30⌋.

* **One person per cell (REQUIRED):** if two centres fall in the same cell, the more visible box is the target (ties: the larger box); the other isn't a target in that frame.
* **Centre outside the frame (REQUIRED):** the box falls in no cell, so it isn't a target.
* **`w` or `h` above 1 (REQUIRED):** the cell's *person here?* target is still 1, but the box is left out of the box loss (a sigmoid can't output more than 1).
* The data itself is unchanged, and every box still counts when TrackEval scores the tracks.

Targets are built on the GPU inside the training loop from each batch's `boxes` and `ids`. The visibility (for the collision rule) is looked up from `boxes.csv`, in the same order as the dataset's boxes.

In [ ]:
# The datasets (consecutive frame pairs) and every frame's visibility values.
TRAIN_PAIRS, VAL_PAIRS, TEST_PAIRS = (MOT17FramePairs(s) for s in ["train", "val", "test"])
boxes_csv = read_prepared("boxes")
VISIBILITY = {key: torch.tensor(g["visibility"].to_numpy(), dtype=torch.float32)
              for key, g in boxes_csv.groupby(["video", "frame"])}
print({s: len(d) for s, d in zip(["train", "val", "test"], [TRAIN_PAIRS, VAL_PAIRS, TEST_PAIRS])}, "pairs")


def build_targets(boxes_list, ids_list, vis_list, count=False):
    """Grid targets for a list of frames (tensors on the GPU).
    Returns person [F,17,30] (0/1), box [F,17,30,4], box_valid [F,17,30], cell_id [F,17,30] (0 = empty),
    and, if count=True, how many boxes were assigned / lost to a collision / outside / left out of the box loss."""
    n = len(boxes_list)
    person = torch.zeros(n, GRID_ROWS, GRID_COLS, device=DEVICE)
    box = torch.zeros(n, GRID_ROWS, GRID_COLS, 4, device=DEVICE)
    box_valid = torch.zeros(n, GRID_ROWS, GRID_COLS, dtype=torch.bool, device=DEVICE)
    cell_id = torch.zeros(n, GRID_ROWS, GRID_COLS, dtype=torch.long, device=DEVICE)
    counts = {"boxes": 0, "assigned": 0, "lost_to_collision": 0, "centre_outside": 0, "left_out_of_box_loss": 0}
    for f, (b, ids, vis) in enumerate(zip(boxes_list, ids_list, vis_list)):
        inside = (b[:, 0] >= 0) & (b[:, 0] < 1) & (b[:, 1] >= 0) & (b[:, 1] < 1)
        if count:
            counts["boxes"] += len(b)
            counts["centre_outside"] += int((~inside).sum())
        b, ids, vis = b[inside], ids[inside], vis[inside]
        if len(b) == 0:
            continue
        row = (b[:, 1] * GRID_ROWS).floor().long()
        col = (b[:, 0] * GRID_COLS).floor().long()
        cell = row * GRID_COLS + col
        # Order the boxes so the winner of each cell comes first: by cell, then visibility (high first),
        # then size (large first). Stable sorts, least important key first.
        order = torch.argsort(b[:, 2] * b[:, 3], descending=True, stable=True)
        order = order[torch.argsort(vis[order], descending=True, stable=True)]
        order = order[torch.argsort(cell[order], stable=True)]
        first_in_cell = torch.ones(len(order), dtype=torch.bool, device=DEVICE)
        first_in_cell[1:] = cell[order][1:] != cell[order][:-1]
        win = order[first_in_cell]
        r, c = row[win], col[win]
        person[f, r, c] = 1.0
        box[f, r, c] = b[win]
        valid = (b[win, 2] <= 1) & (b[win, 3] <= 1)          # boxes wider/taller than the frame: no box loss
        box_valid[f, r, c] = valid
        cell_id[f, r, c] = ids[win]
        if count:
            counts["assigned"] += len(win)
            counts["lost_to_collision"] += len(b) - len(win)
            counts["left_out_of_box_loss"] += int((~valid).sum())
    return {"person": person, "box": box, "box_valid": box_valid, "cell_id": cell_id}, counts


def batch_targets(*batches, count=False):
    """Targets for the frames of one or more batches (in order), from their boxes, IDs and visibility."""
    boxes_list, ids_list, vis_list = [], [], []
    for batch in batches:
        boxes_list += batch["boxes"]
        ids_list += batch["ids"]
        vis_list += [VISIBILITY.get((v, f), torch.zeros(0)).to(DEVICE)
                     for v, f in zip(batch["videos"], batch["frames"])]
    return build_targets(boxes_list, ids_list, vis_list, count)

In [ ]:
# Build the targets of every frame of each split (from the boxes only; no images needed) and count them.
target_counts, assigned_rows = [], []
for split, pairs in [("train", TRAIN_PAIRS), ("val", VAL_PAIRS), ("test", TEST_PAIRS)]:
    ds = pairs.frames
    keys = list(zip(ds.frames["video"], ds.frames["frame"]))
    total = {}
    for start in range(0, len(keys), 256):
        chunk = keys[start:start + 256]
        boxes_list = [ds.targets.get(k, (torch.zeros(0, 4), None))[0].to(DEVICE) for k in chunk]
        ids_list = [ds.targets.get(k, (None, torch.zeros(0, dtype=torch.long)))[1].to(DEVICE) for k in chunk]
        vis_list = [VISIBILITY.get(k, torch.zeros(0)).to(DEVICE) for k in chunk]
        t, counts = build_targets(boxes_list, ids_list, vis_list, count=True)
        total = {k: total.get(k, 0) + v for k, v in counts.items()}
        f, r, c = torch.nonzero(t["person"], as_tuple=True)                 # every assigned cell
        boxes_fc = t["box"][f, r, c].cpu().numpy()
        assigned_rows.append(pd.DataFrame({
            "split": split, "video": [chunk[i][0] for i in f.tolist()], "frame": [chunk[i][1] for i in f.tolist()],
            "id": t["cell_id"][f, r, c].cpu().numpy(), "cx": boxes_fc[:, 0], "cy": boxes_fc[:, 1],
            "w": boxes_fc[:, 2], "h": boxes_fc[:, 3]}))
    total["frames"] = len(keys)
    total["person_cells_per_frame"] = total["assigned"] / len(keys)
    target_counts.append({"split": split, **total})
target_counts = pd.DataFrame(target_counts).set_index("split")
assigned_rows = pd.concat(assigned_rows, ignore_index=True)
display(target_counts.round(2))

# DECISION pos_weight: empty cells / person cells in the training targets (so "nobody anywhere" isn't a good answer).
train_cells = target_counts.loc["train", "frames"] * GRID_ROWS * GRID_COLS
POS_WEIGHT = torch.tensor((train_cells - target_counts.loc["train", "assigned"]) / target_counts.loc["train", "assigned"],
                          device=DEVICE)
print(f"pos_weight = {POS_WEIGHT.item():.3f} (empty cells / person cells in the training targets)")

## M.2 The model (exactly Rowan's description)
* **CNN:** 3 blocks of [3 × (Conv 3×3, padding 1 → ReLU)] → max pooling 4 × 4. Three 4 × 4 poolings take 1088 × 1920 to exactly 17 × 30 (1088 = 17 × 64, 1920 = 30 × 64).
* **FC1 → sigmoid, FC2 → sigmoid, FC3 → sigmoid:** 2,550 = 17 × 30 × 5 outputs: per cell *person here?*, `cx`, `cy`, `w`, `h` (fractions of the whole frame, like `boxes.csv`).
* **Tokenization:** each kept cell's 5 outputs → Linear → 64 numbers. Kept cells: the ground-truth person cells in training (M.1), cells with *person here?* > 0.5 when tracking. No positional encoding (the token already contains `cx`, `cy`).
* **Transformer:** 3 × `nn.TransformerEncoderLayer(d_model=64, nhead=4, dim_feedforward=128, dropout=0.0)`, default ReLU inside. **Padding (REQUIRED):** frames have different numbers of tokens, so they are padded and masked.
* **Comparison:** frame t's transformer outputs · [new person, frame t−1's outputs] ÷ √64, softmax. **New person vector (REQUIRED):** someone who just entered needs a correct answer that isn't an existing ID.

`ReLU(inplace=True)` only saves GPU memory; it computes the same values.

In [ ]:
class TrackerNet(nn.Module):
    def __init__(self):
        super().__init__()
        layers, channels = [], [3, 16, 32, 64]
        for block in range(3):
            for i in range(3):
                in_channels = channels[block] if i == 0 else channels[block + 1]
                layers += [nn.Conv2d(in_channels, channels[block + 1], kernel_size=3, padding=1), nn.ReLU(inplace=True)]
            layers.append(nn.MaxPool2d(kernel_size=4, stride=4))
        self.cnn = nn.Sequential(*layers)
        self.fc1 = nn.Linear(64 * GRID_ROWS * GRID_COLS, 1024)
        self.fc2 = nn.Linear(1024, 1024)
        self.fc3 = nn.Linear(1024, GRID_ROWS * GRID_COLS * CELL_OUTPUTS)
        self.token = nn.Linear(CELL_OUTPUTS, TOKEN_SIZE)
        self.transformer = nn.ModuleList([
            nn.TransformerEncoderLayer(d_model=TOKEN_SIZE, nhead=4, dim_feedforward=128, dropout=0.0, batch_first=True)
            for _ in range(3)])
        self.new_person = nn.Parameter(torch.zeros(TOKEN_SIZE))      # learned "new person" vector

    def grid_logits(self, images):
        """Frames [B, 3, 1088, 1920] -> FC3's values BEFORE its sigmoid, as [B, 17, 30, 5].
        (The losses use these with BCEWithLogitsLoss = sigmoid + binary cross-entropy, computed safely.)"""
        x = self.cnn(images).flatten(1)
        x = torch.sigmoid(self.fc1(x))
        x = torch.sigmoid(self.fc2(x))
        return self.fc3(x).view(-1, GRID_ROWS, GRID_COLS, CELL_OUTPUTS)

    def encode(self, cell_outputs, counts):
        """Kept cells' 5 outputs (after the sigmoid), all frames one after another ([T, 5]), and the number of
        kept cells per frame -> one [n_i, 64] tensor of transformer outputs per frame."""
        tokens = list(torch.split(self.token(cell_outputs), counts))
        frames_with_tokens = [i for i, n in enumerate(counts) if n > 0]   # a frame with no tokens is left as it is
        if frames_with_tokens:
            x = pad_sequence([tokens[i] for i in frames_with_tokens], batch_first=True)    # [frames, longest, 64]
            lengths = torch.tensor([counts[i] for i in frames_with_tokens], device=x.device)
            padding = torch.arange(x.shape[1], device=x.device)[None, :] >= lengths[:, None]   # True = padding
            for layer in self.transformer:
                x = layer(x, src_key_padding_mask=padding)
            for j, i in enumerate(frames_with_tokens):
                tokens[i] = x[j, :counts[i]]
        return tokens

    def match_logits(self, current, previous):
        """Frame t's outputs [n, 64] vs [new person, frame t-1's outputs [m, 64]] -> [n, 1 + m] scores
        (dot product / sqrt(64)); a softmax over each row gives the probabilities. Column 0 = new person."""
        options = torch.cat([self.new_person[None, :], previous])
        return current @ options.T / TOKEN_SIZE ** 0.5


def cell_outputs_of(logits, kept):
    """FC3's sigmoid outputs of the kept cells (frames in order) and the number of kept cells per frame."""
    return torch.sigmoid(logits[kept]), kept.sum(dim=(1, 2)).tolist()

In [ ]:
# Architecture table: real output shapes from one dummy frame on the GPU, and the weights of every layer.
EXPECTED_ARCHITECTURE = [   # (layer, output per frame, weights) from the plan's M.2 table
    ("Conv 1 + ReLU", (16, 1088, 1920), 448), ("Conv 2 + ReLU", (16, 1088, 1920), 2320),
    ("Conv 3 + ReLU", (16, 1088, 1920), 2320), ("Max pool 1 (4x4)", (16, 272, 480), 0),
    ("Conv 4 + ReLU", (32, 272, 480), 4640), ("Conv 5 + ReLU", (32, 272, 480), 9248),
    ("Conv 6 + ReLU", (32, 272, 480), 9248), ("Max pool 2 (4x4)", (32, 68, 120), 0),
    ("Conv 7 + ReLU", (64, 68, 120), 18496), ("Conv 8 + ReLU", (64, 68, 120), 36928),
    ("Conv 9 + ReLU", (64, 68, 120), 36928), ("Max pool 3 (4x4)", (64, 17, 30), 0),
    ("Flatten", (32640,), 0), ("FC1 + sigmoid", (1024,), 33424384), ("FC2 + sigmoid", (1024,), 1049600),
    ("FC3 + sigmoid", (2550,), 2613750), ("Tokenization (Linear 5 -> 64)", (64,), 384),
    ("Transformer encoder 1", (64,), 33472), ("Transformer encoder 2", (64,), 33472),
    ("Transformer encoder 3", (64,), 33472), ("New person vector", (64,), 64),
]
EXPECTED_TOTAL = 37_309_174

seed_everything()
model = TrackerNet().to(DEVICE)
rows = []
with torch.no_grad():
    x = torch.zeros(1, 3, 1088, 1920, device=DEVICE)
    conv_number, pool_number = 0, 0
    for module in model.cnn:
        x = module(x)
        if isinstance(module, nn.Conv2d):
            conv_number += 1
            rows.append([f"Conv {conv_number} + ReLU", tuple(x.shape[1:]), sum(p.numel() for p in module.parameters())])
        elif isinstance(module, nn.MaxPool2d):
            pool_number += 1
            rows.append([f"Max pool {pool_number} (4x4)", tuple(x.shape[1:]), 0])
        else:   # the ReLU after a convolution: same shape, no weights
            rows[-1][1] = tuple(x.shape[1:])
    x = x.flatten(1)
    rows.append(["Flatten", tuple(x.shape[1:]), 0])
    for name, layer in [("FC1 + sigmoid", model.fc1), ("FC2 + sigmoid", model.fc2), ("FC3 + sigmoid", model.fc3)]:
        x = torch.sigmoid(layer(x))
        rows.append([name, tuple(x.shape[1:]), sum(p.numel() for p in layer.parameters())])
    cells = x.view(-1, GRID_ROWS, GRID_COLS, CELL_OUTPUTS)[0, 0, :3]      # 3 example kept cells
    t = model.token(cells)
    rows.append(["Tokenization (Linear 5 -> 64)", tuple(t.shape[1:]), sum(p.numel() for p in model.token.parameters())])
    t = t[None]
    for i, layer in enumerate(model.transformer):
        t = layer(t)
        rows.append([f"Transformer encoder {i + 1}", tuple(t.shape[2:]), sum(p.numel() for p in layer.parameters())])
    rows.append(["New person vector", tuple(model.new_person.shape), model.new_person.numel()])
architecture = pd.DataFrame(rows, columns=["layer", "output per frame (per token after FC3)", "weights"])
architecture.loc[len(architecture)] = ["Total", "", sum(p.numel() for p in model.parameters())]
with pd.option_context("display.max_rows", 50):
    display(architecture.style.format({"weights": "{:,}"}))
TOTAL_WEIGHTS = int(architecture["weights"].iloc[-1])
print(f"Total weights: {TOTAL_WEIGHTS:,} (plan: {EXPECTED_TOTAL:,})")
del model, x, t
torch.cuda.empty_cache()

## M.3 Loss (cross-entropy only) and training
**Loss per pair = person loss + box loss + ID loss** (each weighted 1):
* **Person:** binary cross-entropy on *person here?* over all 510 cells of both frames (averaged), with `pos_weight` from M.1.
* **Box:** binary cross-entropy on `cx, cy, w, h` of the cells with `box_valid`, both frames (averaged).
* **ID:** cross-entropy on frame t's tokens against [new person, frame t−1's tokens]. The answer is the same person in frame t−1, or "new person" (index 0) if they aren't among frame t−1's tokens. A frame with no tokens adds 0.
* Both binary cross-entropies use `BCEWithLogitsLoss` on FC3's values before the sigmoid: the same as sigmoid + binary cross-entropy, without `log(0)`.

**Gradient accumulation (REQUIRED):** 64 pairs (128 full-size frames) don't fit in 12 GB, so `MICRO_BATCH` pairs go through at a time and their gradients are added up; each micro-batch's summed loss is divided by the number of pairs in its step, so one Adam step gets exactly the gradient of the whole batch (the epoch's last step has 3,907 mod batch size pairs).

Validation and test losses use the same ground-truth cells as training (so the three curves measure the same thing); tracking (M.4) uses the cells the model itself finds.

In [ ]:
def pair_losses(model, prev, cur):
    """Losses of each pair in a micro-batch (frame t-1 batch `prev`, frame t batch `cur`, on the GPU).
    Returns person, box and ID loss per pair, and per pair how many of frame t's tokens were matched correctly / in all."""
    n_pairs = len(prev["frames"])
    targets, _ = batch_targets(prev, cur)                       # frames: all t-1 frames, then all t frames
    logits = model.grid_logits(torch.cat([prev["images"], cur["images"]]))     # [2P, 17, 30, 5]

    # Person loss: mean over the 510 cells of each frame, then over the pair's two frames.
    person = F.binary_cross_entropy_with_logits(logits[..., 0], targets["person"], pos_weight=POS_WEIGHT,
                                                reduction="none").mean(dim=(1, 2))
    person = (person[:n_pairs] + person[n_pairs:]) / 2

    # Box loss: mean over the 4 numbers of every valid cell of both frames.
    per_cell = F.binary_cross_entropy_with_logits(logits[..., 1:], targets["box"], reduction="none").mean(dim=-1)
    valid = targets["box_valid"].float()
    loss_sum, cells = (per_cell * valid).sum(dim=(1, 2)), valid.sum(dim=(1, 2))
    loss_sum, cells = loss_sum[:n_pairs] + loss_sum[n_pairs:], cells[:n_pairs] + cells[n_pairs:]
    box = torch.where(cells > 0, loss_sum / cells.clamp(min=1), torch.zeros_like(loss_sum))

    # ID loss: tokens from the ground-truth person cells, through the transformer, compared t vs t-1.
    kept = targets["person"] > 0
    outputs = model.encode(*cell_outputs_of(logits, kept))
    ids = torch.split(targets["cell_id"][kept], [len(o) for o in outputs])
    id_losses, correct, total = [], [], []
    for p in range(n_pairs):
        previous, current = outputs[p], outputs[n_pairs + p]
        if len(current) == 0:
            id_losses.append(logits.new_zeros(()))
            correct.append(0)
            total.append(0)
            continue
        same = ids[n_pairs + p][:, None] == ids[p][None, :]                        # [n, m]
        answer = torch.where(same.any(dim=1), same.float().argmax(dim=1) + 1, 0)   # 0 = new person
        scores = model.match_logits(current, previous)
        id_losses.append(F.cross_entropy(scores, answer))
        correct.append((scores.argmax(dim=1) == answer).sum())
        total.append(len(answer))
    correct = torch.stack([torch.as_tensor(c, device=DEVICE) for c in correct])
    return person, box, torch.stack(id_losses), correct, torch.tensor(total, device=DEVICE)


def backward_micro_batch(model, prev, cur, pairs_in_step):
    """Forward + backward for one micro-batch. Its summed loss is divided by the number of pairs in the whole
    step, so the gradients added up over the step equal those of one big batch."""
    person, box, ident, correct, total = pair_losses(model, prev, cur)
    pair_total = person + box + ident
    (pair_total.sum() / pairs_in_step).backward()
    sums = torch.stack([pair_total.sum(), person.sum(), box.sum(), ident.sum()]).detach()
    return sums, correct.sum(), total.sum()

In [ ]:
# Loaders: the same batching, shuffling (training only) and pin_memory as mot17_data.make_loader, but the worker
# processes stay alive between epochs (starting 8 workers on Windows takes about 30 s, 2 loaders x every epoch).
TRAIN_LOADER = DataLoader(TRAIN_PAIRS, batch_size=MICRO_BATCH, shuffle=True, collate_fn=collate_pairs,
                          num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)
EVAL_PAIRS = ConcatDataset([VAL_PAIRS, TEST_PAIRS])           # validation pairs first, then test pairs
N_VAL = len(VAL_PAIRS)
EVAL_LOADER = DataLoader(EVAL_PAIRS, batch_size=MICRO_BATCH, shuffle=False, collate_fn=collate_pairs,
                         num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=True)


def train_one_epoch(model, optimizer, batch_size, show_memory=False):
    """One pass over the training pairs. One Adam step per `batch_size` pairs (gradient accumulation).
    Returns the losses averaged over the epoch's steps and the ID accuracy over all of frame t's tokens."""
    model.train()
    n_pairs = len(TRAIN_PAIRS)
    step_means, step_sum, seen = [], torch.zeros(4, device=DEVICE), 0
    correct, total = torch.zeros((), device=DEVICE), torch.zeros((), device=DEVICE)
    optimizer.zero_grad(set_to_none=True)
    for prev, cur in TRAIN_LOADER:
        prev, cur = batch_to_device((prev, cur), DEVICE)
        assert prev["images"].is_cuda and cur["images"].is_cuda, "training batch is not on the GPU"
        step = seen // batch_size
        pairs_in_step = min(batch_size, n_pairs - step * batch_size)
        sums, c, t = backward_micro_batch(model, prev, cur, pairs_in_step)
        step_sum += sums
        correct += c
        total += t
        seen += len(prev["frames"])
        if seen == step * batch_size + pairs_in_step:       # every pair of this step is in: one Adam step
            optimizer.step()
            optimizer.zero_grad(set_to_none=True)
            step_means.append(step_sum / pairs_in_step)
            step_sum = torch.zeros(4, device=DEVICE)
            if show_memory and len(step_means) == 1:
                print(f"GPU memory peak after the first step: {torch.cuda.max_memory_allocated() / 1e9:.2f} GB "
                      f"of {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    assert seen == n_pairs, (seen, n_pairs)
    m = torch.stack(step_means).mean(dim=0).tolist()
    return {"loss": m[0], "person": m[1], "box": m[2], "id": m[3], "id_acc": (correct / total).item()}


@torch.no_grad()
def evaluate(model):
    """Full pass over the validation and test pairs (model.eval(), no gradients, on the GPU)."""
    model.eval()
    sums = {s: torch.zeros(4, device=DEVICE) for s in ["val", "test"]}
    correct = {s: torch.zeros((), device=DEVICE) for s in ["val", "test"]}
    total = {s: torch.zeros((), device=DEVICE) for s in ["val", "test"]}
    index = 0
    for prev, cur in EVAL_LOADER:
        prev, cur = batch_to_device((prev, cur), DEVICE)
        assert prev["images"].is_cuda and cur["images"].is_cuda, "validation/test batch is not on the GPU"
        person, box, ident, c, t = pair_losses(model, prev, cur)
        is_val = torch.arange(index, index + len(person), device=DEVICE) < N_VAL
        index += len(person)
        per_pair = torch.stack([person + box + ident, person, box, ident], dim=1)     # [P, 4]
        for split, mask in [("val", is_val), ("test", ~is_val)]:
            sums[split] += per_pair[mask].sum(dim=0)
            correct[split] += c[mask].sum()
            total[split] += t[mask].sum()
    result = {}
    for split, n in [("val", N_VAL), ("test", len(EVAL_PAIRS) - N_VAL)]:
        m = (sums[split] / n).tolist()
        result[split] = {"loss": m[0], "person": m[1], "box": m[2], "id": m[3],
                         "id_acc": (correct[split] / total[split]).item()}
    return result

In [ ]:
def run_name(batch_size, lr):
    return f"bs{batch_size}_lr{lr:g}"

def history_path(batch_size, lr):
    return HISTORY / f"{run_name(batch_size, lr)}.csv"

def checkpoint_path(batch_size, lr, epoch):
    return CHECKPOINTS / f"{run_name(batch_size, lr)}_ep{epoch}.pt"

def run_finished(batch_size, lr):
    path = history_path(batch_size, lr)
    return path.exists() and len(pd.read_csv(path)) >= MAX_EPOCHS


def train_run(batch_size, lr, on_first_epoch=None):
    """Train one setting from scratch for MAX_EPOCHS epochs (seed 0), recording train / validation / test
    metrics every epoch and saving checkpoints at SAVE_EPOCHS. A finished run is skipped (its history is read)."""
    name = run_name(batch_size, lr)
    if run_finished(batch_size, lr):
        print(f"{name}: already finished, skipped")
        return pd.read_csv(history_path(batch_size, lr))
    seed_everything()
    model = TrackerNet().to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    torch.cuda.reset_peak_memory_stats()
    rows = []
    for epoch in range(1, MAX_EPOCHS + 1):
        start = time.time()
        train = train_one_epoch(model, optimizer, batch_size, show_memory=(epoch == 1))
        evaluated = evaluate(model)
        row = {"epoch": epoch, "seconds": time.time() - start}
        for split, metrics in [("train", train), ("val", evaluated["val"]), ("test", evaluated["test"])]:
            row.update({f"{split}_{k}": v for k, v in metrics.items()})
        rows.append(row)
        pd.DataFrame(rows).to_csv(history_path(batch_size, lr), index=False)
        if epoch in SAVE_EPOCHS:
            torch.save(model.state_dict(), checkpoint_path(batch_size, lr, epoch))
        log(f"{name} epoch {epoch}/{MAX_EPOCHS}: {row['seconds']:.0f} s | loss train {row['train_loss']:.4f} "
            f"val {row['val_loss']:.4f} test {row['test_loss']:.4f} | ID acc train {row['train_id_acc']:.3f} "
            f"val {row['val_id_acc']:.3f}")
        if epoch == 1 and on_first_epoch is not None:
            on_first_epoch(row["seconds"])
    del model, optimizer
    torch.cuda.empty_cache()
    return pd.DataFrame(rows)


def load_model(batch_size, lr, epoch):
    model = TrackerNet().to(DEVICE)
    model.load_state_dict(torch.load(checkpoint_path(batch_size, lr, epoch), map_location=DEVICE))
    return model.eval()

## M.4 Tracking (validation and test videos)
Each video runs frame by frame in its original order. The previous frame's transformer outputs and IDs are kept (nothing older).
* **Kept cells:** *person here?* > 0.5. On a video's first frame every kept cell gets a new ID.
* **One ID per person per frame (REQUIRED):** each box takes its most likely option; if two boxes choose the same previous person, the one with the higher probability keeps it and the other takes its next most likely option (possibly "new person"). This greedy rule is not Hungarian matching; it only prevents duplicate IDs, which TrackEval rejects.
* Boxes go back to original pixels with `boxes_to_pixels` and are written in MOTChallenge format (conf = *person here?*). Scored with TrackEval using exactly the baseline notebook's settings.

In [ ]:
def assign_ids(probabilities):
    """[n, 1 + m] softmax probabilities -> chosen option per box (0 = new person, j >= 1 = previous person j - 1).
    Repeatedly takes the most probable (box, option) left; a previous person, once taken, can't be chosen again."""
    p = probabilities.clone()
    n = p.shape[0]
    choice = torch.full((n,), -1, dtype=torch.long)
    done = torch.zeros(n, dtype=torch.bool)
    for _ in range(n):
        best_p, best_option = p.max(dim=1)
        best_p[done] = -1.0
        i = int(best_p.argmax())
        j = int(best_option[i])
        choice[i], done[i] = j, True
        if j >= 1:
            p[:, j] = -1.0                      # that previous person is taken
    return choice


@torch.no_grad()
def track_split(model, split, tracker_name):
    """Track every video of a split; write MOTChallenge files to trackers/<tracker_name>/data/.
    Returns a table of every output box: video, frame, id, person probability, cx, cy, w, h (fractions)."""
    model.eval()
    dataset = MOT17Frames(split)
    loader = DataLoader(dataset, batch_size=2 * MICRO_BATCH, shuffle=False, collate_fn=collate_frames,
                        num_workers=NUM_WORKERS, pin_memory=True)
    records = []
    previous, previous_ids, current_video, next_id = None, None, None, 1
    for batch in loader:
        batch = batch_to_device(batch, DEVICE)
        assert batch["images"].is_cuda, "tracking batch is not on the GPU"
        outputs = torch.sigmoid(model.grid_logits(batch["images"]))              # [B, 17, 30, 5]
        kept = outputs[..., 0] > PERSON_THRESHOLD
        cells, counts = outputs[kept], kept.sum(dim=(1, 2)).tolist()
        encoded = model.encode(cells, counts)
        cells = torch.split(cells, counts)
        for i, (video, frame) in enumerate(zip(batch["videos"], batch["frames"])):
            if video != current_video:                         # a new video: forget the previous frame
                previous, current_video, next_id = None, video, 1
            n = counts[i]
            ids = torch.zeros(n, dtype=torch.long)
            if previous is not None and len(previous) > 0 and n > 0:
                probabilities = torch.softmax(model.match_logits(encoded[i], previous), dim=1).cpu()
                choice = assign_ids(probabilities)
                matched = choice >= 1
                ids[matched] = previous_ids[choice[matched] - 1]
            for k in torch.nonzero(ids == 0).flatten().tolist():      # new people (and everyone on frame 1)
                ids[k] = next_id
                next_id += 1
            c = cells[i].cpu().numpy()
            for k in range(n):
                records.append((video, frame, int(ids[k]), float(c[k, 0]), *map(float, c[k, 1:])))
            previous, previous_ids = encoded[i], ids
    records = pd.DataFrame(records, columns=["video", "frame", "id", "conf", "cx", "cy", "w", "h"])

    # MOTChallenge files, boxes in original pixels.
    for video in dataset.frames["video"].unique():
        info = VIDEOS.set_index("video").loc[video]
        r = records[records["video"] == video].sort_values(["frame", "id"])
        px = boxes_to_pixels(r, info["im_width"], info["im_height"])
        lines = [f"{f},{i},{l:.2f},{t:.2f},{w:.2f},{h:.2f},{c:.4f},-1,-1,-1"
                 for f, i, l, t, w, h, c in zip(r["frame"], r["id"], px["left"], px["top"], px["width"],
                                                 px["height"], r["conf"])]
        path = TRACKERS / tracker_name / "data" / f"{video}-{DETECTOR_COPY}.txt"
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_text("\n".join(lines) + ("\n" if lines else ""))
    return records

In [ ]:
# TrackEval with exactly the DeepSORT baseline notebook's settings.
np.float = float     # TrackEval still uses these two names, removed from NumPy (they were only aliases)
np.int = int
import trackeval     # prints a note about BURST/pycocotools: a different dataset, not used here

EVAL_VIDEOS = VIDEOS[VIDEOS["split"].isin(["val", "test"])]["video"].tolist()


def score(tracker_names):
    """HOTA, MOTA, IDF1, ID switches, false positives and false negatives per tracker and video."""
    eval_config = trackeval.Evaluator.get_default_eval_config()
    eval_config.update({"USE_PARALLEL": False, "PRINT_CONFIG": False, "PRINT_RESULTS": False,
                        "TIME_PROGRESS": False, "PLOT_CURVES": False, "LOG_ON_ERROR": str(EVAL / "error_log.txt")})
    dataset_config = trackeval.datasets.MotChallenge2DBox.get_default_dataset_config()
    dataset_config.update({
        "GT_FOLDER": str(DATA_ROOT / "train"), "TRACKERS_FOLDER": str(TRACKERS), "TRACKERS_TO_EVAL": tracker_names,
        "OUTPUT_FOLDER": str(EVAL), "CLASSES_TO_EVAL": ["pedestrian"], "BENCHMARK": "MOT17", "SPLIT_TO_EVAL": "train",
        "SKIP_SPLIT_FOL": True, "SEQ_INFO": {f"{v}-{DETECTOR_COPY}": None for v in EVAL_VIDEOS},
        "TRACKER_SUB_FOLDER": "data", "DO_PREPROC": True, "PRINT_CONFIG": False})
    metric_config = {"THRESHOLD": 0.5, "PRINT_CONFIG": False}
    metrics = [trackeval.metrics.HOTA(metric_config), trackeval.metrics.CLEAR(metric_config),
               trackeval.metrics.Identity(metric_config)]
    results, messages = trackeval.Evaluator(eval_config).evaluate(
        [trackeval.datasets.MotChallenge2DBox(dataset_config)], metrics)
    assert all(m == "Success" for m in messages["MotChallenge2DBox"].values()), messages
    rows = []
    for name in tracker_names:
        for video in EVAL_VIDEOS:
            r = results["MotChallenge2DBox"][name][f"{video}-{DETECTOR_COPY}"]["pedestrian"]
            rows.append({"tracker": name, "video": video, "HOTA": 100 * float(r["HOTA"]["HOTA"].mean()),
                         "MOTA": 100 * float(r["CLEAR"]["MOTA"]), "IDF1": 100 * float(r["Identity"]["IDF1"]),
                         "ID_switches": int(r["CLEAR"]["IDSW"]), "false_positives": int(r["CLEAR"]["CLR_FP"]),
                         "false_negatives": int(r["CLEAR"]["CLR_FN"])})
    return pd.DataFrame(rows)

## M.5 Checks before training
Each check prints PASS or FAIL with what it counted. The notebook stops here if any fails.

In [ ]:
check_results = []

def check(name, passed, detail):
    check_results.append({"stage": "before training" if "after" not in name else "after training",
                          "check": name, "passed": bool(passed), "detail": detail})
    print(f"[{'PASS' if passed else 'FAIL'}] {name}: {detail}")


# 1. GPU
seed_everything()
model = TrackerNet().to(DEVICE)
check("GPU is the RTX 5070", "RTX 5070" in GPU_NAME, GPU_NAME)
check("every parameter is on cuda", all(p.is_cuda for p in model.parameters()),
      f"{sum(1 for _ in model.parameters())} parameter tensors")
asserting = [f.__name__ for f in [train_one_epoch, evaluate, track_split] if "is_cuda" in inspect.getsource(f)]
check("train, validation/test and tracking functions assert their batch is on cuda", len(asserting) == 3,
      ", ".join(asserting))

# 2. Architecture
shapes_ok = all(tuple(a[1]) == tuple(e[1]) and a[2] == e[2] and a[0] == e[0]
                for a, e in zip(architecture.iloc[:-1].itertuples(index=False), EXPECTED_ARCHITECTURE))
check("architecture matches the M.2 table", shapes_ok and len(architecture) - 1 == len(EXPECTED_ARCHITECTURE)
      and TOTAL_WEIGHTS == EXPECTED_TOTAL, f"{len(EXPECTED_ARCHITECTURE)} layers; total {TOTAL_WEIGHTS:,} weights "
      f"(expected {EXPECTED_TOTAL:,})")

# 3. Targets: every box is accounted for, and the assigned targets are the boxes.csv boxes.
for split, r in target_counts.iterrows():
    accounted = r["assigned"] + r["lost_to_collision"] + r["centre_outside"]
    check(f"{split} targets account for every box", accounted == r["boxes"],
          f"{int(r['assigned']):,} assigned + {int(r['lost_to_collision']):,} lost to collisions + "
          f"{int(r['centre_outside']):,} centre outside = {int(accounted):,} of {int(r['boxes']):,} boxes")
BOX_COLUMNS = ["cx", "cy", "w", "h"]
compare = assigned_rows.merge(boxes_csv, on=["split", "video", "frame", "id"], suffixes=("_target", ""))
size = VIDEOS.set_index("video").loc[compare["video"], ["im_width", "im_height"]].to_numpy()
targets_only = pd.DataFrame({c: compare[c + "_target"] for c in BOX_COLUMNS})
target_px = boxes_to_pixels(targets_only, size[:, 0], size[:, 1])        # targets back to original pixels
csv_px = boxes_to_pixels(compare[BOX_COLUMNS], size[:, 0], size[:, 1])    # boxes.csv in original pixels
max_px = float(np.abs(target_px.to_numpy() - csv_px.to_numpy()).max())
max_fraction = float(np.abs(targets_only.to_numpy() - compare[BOX_COLUMNS].to_numpy()).max())
check("assigned targets converted back to pixels match boxes.csv",
      len(compare) == len(assigned_rows) and max_fraction <= 1e-6,
      f"{len(compare):,} of {len(assigned_rows):,} assigned targets found in boxes.csv; largest difference "
      f"{max_px:.6f} px ({max_fraction:.1e} of the frame size; the dataset stores boxes as float32, so this is the "
      "rounding of float32, not an altered box)")
ids_same = all(torch.equal(TRAIN_PAIRS.frames.targets[k][1],
                           torch.tensor(g["id"].to_numpy())) for k, g in boxes_csv[boxes_csv["split"] == "train"].groupby(["video", "frame"]))
check("visibility lookup is in the same box order as the dataset", ids_same, "IDs compared frame by frame (train)")

In [ ]:
# 4. Gradient accumulation: one batch of 8 pairs vs 2 micro-batches of 4, same model, same pairs.
#    (16 full-size frames at once need more than the GPU's 12 GB; Windows then borrows shared system memory,
#    which is slow but fine for this one check.)
def fixed_pairs(indices):
    return batch_to_device(collate_pairs([TRAIN_PAIRS[i] for i in indices]), DEVICE)


def gradients(model):
    return torch.cat([p.grad.flatten() for p in model.parameters() if p.grad is not None])


seed_everything()
model = TrackerNet().to(DEVICE)
model.train()
prev8, cur8 = fixed_pairs(range(0, 800, 100))         # 8 pairs from different parts of the training videos
model.zero_grad(set_to_none=True)
backward_micro_batch(model, prev8, cur8, pairs_in_step=8)
one_batch = gradients(model).clone()
model.zero_grad(set_to_none=True)
for half in [slice(0, 4), slice(4, 8)]:
    prev4 = {k: v[half] for k, v in prev8.items()}
    cur4 = {k: v[half] for k, v in cur8.items()}
    backward_micro_batch(model, prev4, cur4, pairs_in_step=8)
accumulated = gradients(model)
relative = float((one_batch - accumulated).norm() / one_batch.norm())
check("gradient accumulation: 1 x 8 pairs = 2 x 4 pairs", relative <= 1e-5,
      f"relative difference {relative:.2e} (limit 1e-5) over {one_batch.numel():,} gradient values")
del model, prev8, cur8, prev4, cur4, one_batch, accumulated
torch.cuda.empty_cache()

In [ ]:
# 5. Overfitting test: 4 pairs, 200 Adam steps at learning rate 0.001. The loss must fall by more than 90% and
#    ID accuracy on those pairs must reach 100%. (If not: stop and report; no layers are added.)
seed_everything()
model = TrackerNet().to(DEVICE)
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
prev4, cur4 = fixed_pairs(range(4))
losses = []
start = time.time()
for step in range(200):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    sums, _, _ = backward_micro_batch(model, prev4, cur4, pairs_in_step=4)
    optimizer.step()
    losses.append((sums / 4).tolist())
with torch.no_grad():
    model.eval()
    person, box, ident, correct, total = pair_losses(model, prev4, cur4)
final_accuracy = (correct.sum() / total.sum()).item()
# The box loss can't reach 0: binary cross-entropy with a fractional target t is at least the target's entropy.
t, _ = batch_targets(prev4, cur4)
tb = t["box"][t["box_valid"]].clamp(1e-12, 1 - 1e-12)
box_floor = float((-(tb * tb.log() + (1 - tb) * (1 - tb).log())).mean())
losses = pd.DataFrame(losses, columns=["loss", "person", "box", "id"])
fall = 1 - losses["loss"].iloc[-1] / losses["loss"].iloc[0]
print(losses.iloc[[0, 49, 99, 149, 199]].round(4).to_string())
print(f"{time.time() - start:.0f} s; lowest possible box loss on these pairs (target entropy): {box_floor:.4f}")
check("overfitting test: loss falls by more than 90% in 200 steps", fall > 0.9,
      f"loss {losses['loss'].iloc[0]:.4f} -> {losses['loss'].iloc[-1]:.4f} ({100 * fall:.1f}% lower); "
      f"box loss {losses['box'].iloc[-1]:.4f} vs its floor {box_floor:.4f}")
check("overfitting test: ID accuracy 100% on the 4 pairs", final_accuracy == 1.0,
      f"{int(correct.sum())} of {int(total.sum())} tokens of frame t matched correctly")
del model, optimizer, prev4, cur4
torch.cuda.empty_cache()

failed = [c for c in check_results if not c["passed"]]
if failed:
    raise RuntimeError("Stopping before training: " + "; ".join(c["check"] for c in failed))
print("All checks before training passed.")

## M.6 First iteration: batch 64, learning rate 0.001, 16 epochs
This run trains to 24 epochs (checkpoints at 8, 16 and 24) because it is also part of the M.9 grid. **The first-iteration model is the epoch-16 checkpoint**, the same as a separate 16-epoch run (nothing depends on the total number of epochs: there's no learning-rate schedule). The first epoch is timed to estimate this run and the grid.

In [ ]:
TIMING_FILE = HISTORY / "timing.json"


def grid_runs(small=GRID_OPTION_SMALL):
    if small:
        return list(itertools.product([32, 64], [0.0001, 0.001]))
    return list(itertools.product(GRID_BATCH_SIZES, GRID_LEARNING_RATES))


def report_timing(seconds_per_epoch):
    """Called after the first epoch: estimated time of this run and of the grid."""
    TIMING_FILE.write_text(json.dumps({"seconds_per_epoch": seconds_per_epoch, "gpu": GPU_NAME}))
    log(f"first epoch took {seconds_per_epoch / 60:.1f} min -> this run (24 epochs) about "
        f"{24 * seconds_per_epoch / 3600:.1f} h; full grid ({len(grid_runs(False))} runs) about "
        f"{len(grid_runs(False)) * 24 * seconds_per_epoch / 3600:.1f} h; small grid ({len(grid_runs(True))} runs) about "
        f"{len(grid_runs(True)) * 24 * seconds_per_epoch / 3600:.1f} h")


first_history = train_run(BATCH_SIZE, LEARNING_RATE, on_first_epoch=report_timing)
FIRST = {"batch_size": BATCH_SIZE, "lr": LEARNING_RATE, "epochs": EPOCHS}
print(f"First-iteration model: {checkpoint_path(BATCH_SIZE, LEARNING_RATE, EPOCHS).name}")
display(first_history.round(4))

## M.7 Curves
Total loss and ID accuracy against epoch for training, validation and testing, and the three loss parts for training and validation. The test curve is only plotted (Rowan asked for it); it is never used to choose anything.

In [ ]:
def plot_curves(history, last_epoch, title, prefix):
    """The M.7 plots for one run, epochs 1..last_epoch, saved as figures/<prefix>_*.png."""
    h = history[history["epoch"] <= last_epoch]
    paths = []
    for metric, ylabel, scale in [("loss", "total loss", 1), ("id_acc", "ID accuracy (%)", 100)]:
        fig, ax = plt.subplots(figsize=(12, 5))
        for split in ["train", "val", "test"]:
            ax.plot(h["epoch"], scale * h[f"{split}_{metric}"], color=SPLIT_COLOURS[split], linewidth=2,
                    marker="o", markersize=5, label=SPLIT_NAMES[split])
        ax.set_xlabel("epoch")
        ax.set_ylabel(ylabel)
        ax.set_title(f"{title}: {ylabel}")
        ax.grid(color="#e1e0d9")
        ax.legend()
        plt.tight_layout()
        paths.append(FIGURES / f"{prefix}_{metric}.png")
        fig.savefig(paths[-1], dpi=110)
        plt.show()
    fig, ax = plt.subplots(figsize=(12, 5))
    for part, colour in [("person", "#2a78d6"), ("box", "#eb6834"), ("id", "#1baf7a")]:
        for split, style in [("train", "-"), ("val", "--")]:
            ax.plot(h["epoch"], h[f"{split}_{part}"], color=colour, linestyle=style, linewidth=2,
                    label=f"{part} loss, {SPLIT_NAMES[split]}")
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.set_title(f"{title}: loss parts (solid = training, dashed = validation)")
    ax.grid(color="#e1e0d9")
    ax.legend(ncol=3)
    plt.tight_layout()
    paths.append(FIGURES / f"{prefix}_loss_parts.png")
    fig.savefig(paths[-1], dpi=110)
    plt.show()
    return paths


first_curve_files = plot_curves(first_history, EPOCHS, "First iteration (batch 64, lr 0.001)", "first_iteration")

## M.8 Sample image: three stages (MOT17-13, frame 300)
All on the resized 1920 × 1088 frame:
1. **Pre-processed and cleaned:** the model's input frame (normalization undone, only for display) with the cleaned ground-truth boxes from `boxes.csv`. The boxes are the training targets; the model is given only the image.
2. **After the fully connected layers:** the boxes of cells with *person here?* > 0.5, labelled with that probability, no IDs.
3. **Final output:** the same boxes with their IDs (one colour per ID), from tracking MOT17-13 from frame 1 to frame 300.

Each panel is shown large; the three side by side are saved for the report.

In [ ]:
TEST_FRAMES = MOT17Frames("test")
PIXEL_STATS = read_prepared("pixel_stats")


def sample_figure(records, title, file_name):
    """The three-stage figure for SAMPLE_FRAME, from a tracking table (track_split's output)."""
    position = TEST_FRAMES.frames.index[(TEST_FRAMES.frames["video"] == SAMPLE_VIDEO)
                                        & (TEST_FRAMES.frames["frame"] == SAMPLE_FRAME)][0]
    item = TEST_FRAMES[position]
    mean = torch.tensor(PIXEL_STATS["mean"])[:, None, None]
    std = torch.tensor(PIXEL_STATS["std"])[:, None, None]
    image = (item["image"] * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()    # normalization undone for display
    W, H = mot17_data.TARGET_WIDTH, mot17_data.TARGET_HEIGHT
    found = records[(records["video"] == SAMPLE_VIDEO) & (records["frame"] == SAMPLE_FRAME)]

    def draw(ax, stage, scale):
        ax.imshow(image)
        if stage == 1:
            for cx, cy, w, h in item["boxes"].tolist():
                ax.add_patch(Rectangle(((cx - w / 2) * W, (cy - h / 2) * H), w * W, h * H, fill=False,
                                       edgecolor="#2a78d6", linewidth=1.5 * scale))
            text = (f"1. Pre-processed and cleaned: model input {W} x {H} + {len(item['boxes'])} cleaned "
                    "ground-truth boxes (targets; the model gets only the image)")
        else:
            for r in found.itertuples():
                colour = "#eb6834" if stage == 2 else plt.cm.tab20(r.id % 20)
                ax.add_patch(Rectangle(((r.cx - r.w / 2) * W, (r.cy - r.h / 2) * H), r.w * W, r.h * H, fill=False,
                                       edgecolor=colour, linewidth=(1.5 if stage == 2 else 2.5) * scale))
                label = f"{r.conf:.2f}" if stage == 2 else str(r.id)
                ax.text((r.cx - r.w / 2) * W, (r.cy - r.h / 2) * H - 4, label, color=colour,
                        fontsize=7 * scale, fontweight="bold", va="bottom")
            text = (f"2. After the fully connected layers: {len(found)} cells with person here? > {PERSON_THRESHOLD}"
                    if stage == 2 else
                    f"3. Final output: {len(found)} boxes with IDs (tracked from frame 1 to {SAMPLE_FRAME})")
        ax.set_title(text, fontsize=10 * scale)
        ax.axis("off")

    for stage in [1, 2, 3]:                           # shown large, one by one
        fig, ax = plt.subplots(figsize=(16, 9.4))
        draw(ax, stage, 1.4)
        plt.tight_layout()
        plt.show()
    fig, axes = plt.subplots(1, 3, figsize=(30, 6.4))  # saved side by side for the report
    for stage, ax in zip([1, 2, 3], axes):
        draw(ax, stage, 1.0)
    fig.suptitle(f"{title}: {SAMPLE_VIDEO} frame {SAMPLE_FRAME}", fontsize=14)
    fig.tight_layout()
    path = FIGURES / file_name
    fig.savefig(path, dpi=100)
    plt.close(fig)
    print("Saved", path)
    return path


first_model = load_model(BATCH_SIZE, LEARNING_RATE, EPOCHS)
first_tracks = {split: track_split(first_model, split, "first_iteration") for split in ["val", "test"]}
first_sample_file = sample_figure(first_tracks["test"], "First iteration (batch 64, lr 0.001, 16 epochs)",
                                  "first_iteration_sample.png")
del first_model
torch.cuda.empty_cache()

## M.9 Grid search (validation only)
Batch size × learning rate runs, each trained to 24 epochs; epochs 8, 16 and 24 are read from each run. **The best setting is the lowest validation total loss** over all combinations; test numbers are not used. Only the checkpoints of the first-iteration model and the best model so far are kept.

In [ ]:
def grid_table(runs):
    """One row per (batch size, learning rate, epochs) of the finished runs, from their history files."""
    rows = []
    for batch_size, lr in runs:
        if not run_finished(batch_size, lr):
            continue
        h = pd.read_csv(history_path(batch_size, lr)).set_index("epoch")
        for epochs in GRID_EPOCHS:
            r = h.loc[epochs]
            rows.append({"batch_size": batch_size, "lr": lr, "epochs": epochs, "val_loss": r["val_loss"],
                         "val_person": r["val_person"], "val_box": r["val_box"], "val_id": r["val_id"],
                         "val_id_acc": r["val_id_acc"], "train_loss": r["train_loss"]})
    return pd.DataFrame(rows)


def choose_best(table):
    """The combination with the lowest VALIDATION total loss (only validation numbers are looked at)."""
    return table.loc[table["val_loss"].idxmin()]


def keep_checkpoints(best):
    """Delete every checkpoint except the first-iteration model and the best model so far."""
    keep = {checkpoint_path(BATCH_SIZE, LEARNING_RATE, EPOCHS),
            checkpoint_path(int(best["batch_size"]), float(best["lr"]), int(best["epochs"]))}
    for path in CHECKPOINTS.glob("*.pt"):
        if path not in keep:
            path.unlink()


def save_run_curves(batch_size, lr):
    """A small two-panel curve figure for every grid run (loss and ID accuracy: training, validation, testing)."""
    h = pd.read_csv(history_path(batch_size, lr))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for ax, (metric, ylabel, scale) in zip(axes, [("loss", "total loss", 1), ("id_acc", "ID accuracy (%)", 100)]):
        for split in ["train", "val", "test"]:
            ax.plot(h["epoch"], scale * h[f"{split}_{metric}"], color=SPLIT_COLOURS[split], label=SPLIT_NAMES[split])
        ax.set_xlabel("epoch")
        ax.set_ylabel(ylabel)
        ax.grid(color="#e1e0d9")
        ax.legend()
    fig.suptitle(f"Run {run_name(batch_size, lr)}")
    fig.tight_layout()
    fig.savefig(FIGURES / "runs" / f"{run_name(batch_size, lr)}.png", dpi=90)
    plt.close(fig)


RUNS = grid_runs()
seconds_per_epoch = json.loads(TIMING_FILE.read_text())["seconds_per_epoch"]
grid_hours = len(RUNS) * MAX_EPOCHS * seconds_per_epoch / 3600
remaining = [r for r in RUNS if not run_finished(*r)]
print(f"Grid: {len(RUNS)} runs ({'small OPTION' if GRID_OPTION_SMALL else 'full'}), about {grid_hours:.1f} h in all; "
      f"{len(remaining)} runs left, about {len(remaining) * MAX_EPOCHS * seconds_per_epoch / 3600:.1f} h")
if grid_hours > MAX_GRID_HOURS and not ALLOW_LONG_GRID:
    raise RuntimeError(f"The grid would take about {grid_hours:.1f} h (over {MAX_GRID_HOURS} h). Stopping to ask Rowan: "
                       "set GRID_OPTION_SMALL = True, or ALLOW_LONG_GRID = True, in the settings cell.")

for batch_size, lr in RUNS:
    train_run(batch_size, lr)
    save_run_curves(batch_size, lr)
    keep_checkpoints(choose_best(grid_table(RUNS)))

grid = grid_table(RUNS)
best = choose_best(grid)
BEST = {"batch_size": int(best["batch_size"]), "lr": float(best["lr"]), "epochs": int(best["epochs"])}
print(f"{len(grid)} combinations. Best on validation: batch {BEST['batch_size']}, lr {BEST['lr']:g}, "
      f"{BEST['epochs']} epochs (validation loss {best['val_loss']:.4f})")
with pd.option_context("display.max_rows", 40):
    display(grid.sort_values("val_loss").round(4))

## M.10 Final results
Curves, the sample image and TrackEval (validation and test videos) for the first-iteration and the best model, then the comparison with the DeepSORT baseline on MOT17-13 (test) and `model_report.md`.

In [ ]:
best_history = pd.read_csv(history_path(BEST["batch_size"], BEST["lr"]))
best_title = f"Best (batch {BEST['batch_size']}, lr {BEST['lr']:g}, {BEST['epochs']} epochs)"
best_curve_files = plot_curves(best_history, BEST["epochs"], best_title, "best")

best_model = load_model(BEST["batch_size"], BEST["lr"], BEST["epochs"])
best_tracks = {split: track_split(best_model, split, "best") for split in ["val", "test"]}
best_sample_file = sample_figure(best_tracks["test"], best_title, "best_sample.png")
del best_model
torch.cuda.empty_cache()

In [ ]:
scores = score(["first_iteration", "best"])
scores.to_csv(OUT / "results.csv", index=False)
display(scores.round(1))

# Comparison on MOT17-13 (test) with the DeepSORT baseline (our own Faster R-CNN + DeepSORT).
deepsort = pd.read_csv(mot17_data.PROJECT_ROOT / "outputs" / "deepsort" / "results.csv")
deepsort = deepsort[(deepsort["source"] == "frcnn") & (deepsort["video"] == "MOT17-13")].iloc[0]
COLUMNS = ["HOTA", "MOTA", "IDF1", "ID_switches", "false_positives", "false_negatives"]
test_scores = scores[scores["video"] == "MOT17-13"].set_index("tracker")
comparison = pd.DataFrame([
    {"model": "Our model, first iteration (batch 64, lr 0.001, 16 epochs)", **test_scores.loc["first_iteration", COLUMNS]},
    {"model": f"Our model, best on validation (batch {BEST['batch_size']}, lr {BEST['lr']:g}, {BEST['epochs']} epochs)",
     **test_scores.loc["best", COLUMNS]},
    {"model": "DeepSORT baseline (Faster R-CNN + DeepSORT)", **deepsort[COLUMNS]},
])
for c in ["ID_switches", "false_positives", "false_negatives"]:
    comparison[c] = comparison[c].astype(int)
display(comparison.round(1))

In [ ]:
def md_table(df, decimals=1):
    def fmt(v):
        if isinstance(v, (bool, np.bool_)):
            return "yes" if v else "**NO**"
        if isinstance(v, (int, np.integer)):
            return f"{v:,}"
        if isinstance(v, (float, np.floating)):
            return "n/a" if np.isnan(v) else f"{v:,.{decimals}f}"
        return str(v)
    lines = ["| " + " | ".join(map(str, df.columns)) + " |", "|" + "|".join(["---"] * len(df.columns)) + "|"]
    lines += ["| " + " | ".join(fmt(v) for v in row) + " |" for row in df.itertuples(index=False)]
    return "\n".join(lines)


ours, base = comparison.iloc[1], comparison.iloc[2]
first_row = comparison.iloc[0]
reading = (
    f"On the test video MOT17-13, the first-iteration model scores HOTA {first_row['HOTA']:.1f}, MOTA {first_row['MOTA']:.1f} "
    f"and IDF1 {first_row['IDF1']:.1f}; the model chosen on validation (batch {BEST['batch_size']}, lr {BEST['lr']:g}, "
    f"{BEST['epochs']} epochs) scores HOTA {ours['HOTA']:.1f}, MOTA {ours['MOTA']:.1f} and IDF1 {ours['IDF1']:.1f}. "
    f"The DeepSORT baseline scores HOTA {base['HOTA']:.1f}, MOTA {base['MOTA']:.1f} and IDF1 {base['IDF1']:.1f}. "
    f"Our best model has {int(ours['false_positives']):,} false positives and {int(ours['false_negatives']):,} false negatives "
    f"against the baseline's {int(base['false_positives']):,} and {int(base['false_negatives']):,}. "
    f"By design the grid can hold at most one person per 64 x 64 cell: {int(target_counts.loc['test', 'lost_to_collision']):,} "
    f"of the test video's {int(target_counts.loc['test', 'boxes']):,} boxes share a cell with a more visible person, "
    "so they can't be found even by a perfect model of this design.")
print(reading)

settings_table = pd.DataFrame({"setting": ["batch size (first iteration)", "learning rate (first iteration)",
                                           "epochs (first iteration)", "optimizer", "micro-batch", "grid",
                                           "person threshold", "pos_weight", "workers", "seed", "GPU", "data version"],
                               "value": [BATCH_SIZE, LEARNING_RATE, EPOCHS, "Adam, default betas, no weight decay",
                                         f"{MICRO_BATCH} pairs", f"{GRID_ROWS} x {GRID_COLS}", PERSON_THRESHOLD,
                                         f"{POS_WEIGHT.item():.3f}", NUM_WORKERS, SEED, GPU_NAME, DATA_VERSION]})
settings_table["value"] = settings_table["value"].astype(str)
arch_md = architecture.copy()
arch_md["output per frame (per token after FC3)"] = arch_md["output per frame (per token after FC3)"].astype(str)
counts_md = target_counts.reset_index()[["split", "frames", "boxes", "assigned", "lost_to_collision", "centre_outside",
                                         "left_out_of_box_loss", "person_cells_per_frame"]]
report_text = "\n\n".join([
    "# Model report: first iteration and grid search",
    f"Generated by `Model.ipynb`. Data version {DATA_VERSION}. Train MOT17-02, 04, 05, 09, 11; validation MOT17-10; "
    "test MOT17-13 (never used to choose anything).",
    "## Architecture\n\n" + md_table(arch_md),
    "## Settings\n\n" + md_table(settings_table),
    "## Training targets (M.1)\n\n" + md_table(counts_md, decimals=2),
    f"## Grid search (validation total loss; best = lowest)\n\n" + md_table(grid.sort_values("val_loss"), decimals=4),
    "## Curves\n\n" + "\n".join(f"- `figures/{p.name}`" for p in first_curve_files + best_curve_files),
    f"## Sample image ({SAMPLE_VIDEO} frame {SAMPLE_FRAME})\n\n- `figures/{first_sample_file.name}`\n- `figures/{best_sample_file.name}`",
    "## TrackEval (validation and test videos)\n\n" + md_table(scores),
    "## Comparison on MOT17-13 (test)\n\n" + md_table(comparison),
    "## What the results mean\n\n" + reading,
])
(OUT / "model_report.md").write_text(report_text + "\n", encoding="utf-8")
print("Saved", OUT / "model_report.md")

## M.11 Checks after training

In [ ]:
# Tracker files: frames in range, positive whole-number IDs, no ID twice in one frame, boxes in original pixels.
for name in ["first_iteration", "best"]:
    for video in EVAL_VIDEOS:
        info = VIDEOS.set_index("video").loc[video]
        path = TRACKERS / name / "data" / f"{video}-{DETECTOR_COPY}.txt"
        t = pd.read_csv(path, header=None, names=["frame", "id", "left", "top", "width", "height", "conf", "x", "y", "z"]) \
            if path.stat().st_size else pd.DataFrame(columns=["frame", "id", "left", "top", "width", "height"])
        frames_ok = t["frame"].between(1, info["seq_length"]).all()
        ids_ok = (t["id"] > 0).all() and (t["id"] % 1 == 0).all()
        no_duplicates = not t.duplicated(["frame", "id"]).any()
        cx, cy = t["left"] + t["width"] / 2, t["top"] + t["height"] / 2
        pixels_ok = (len(t) == 0 or ((t["width"] > 0).all() and cx.between(0, info["im_width"]).all()
                     and cy.between(0, info["im_height"]).all() and (t[["width", "height"]].to_numpy().max() > 1)))
        check(f"after training: {name} {video} tracker file valid", frames_ok and ids_ok and no_duplicates and pixels_ok,
              f"{len(t):,} rows, {t['id'].nunique()} IDs, duplicate (frame, ID) pairs: {int(t.duplicated(['frame', 'id']).sum())}, "
              f"centres inside the {info['im_width']} x {info['im_height']} frame: {bool(pixels_ok)}")

raw_md5_after = {str(p): mot17_data.file_md5(p) for p in RAW_FILES}
check("after training: raw gt.txt and seqinfo.ini unchanged", raw_md5_after == raw_md5_before,
      f"{len(RAW_FILES)} files compared by MD5")
check("after training: prepared data unchanged", mot17_data.data_version() == DATA_VERSION, f"data version {DATA_VERSION}")

curves_ok = []
for batch_size, lr in RUNS:
    h = pd.read_csv(history_path(batch_size, lr))
    curves_ok.append(len(h) == MAX_EPOCHS and h[[f"{s}_loss" for s in ["train", "val", "test"]]].notna().all().all()
                     and (FIGURES / "runs" / f"{run_name(batch_size, lr)}.png").exists())
check("after training: every run has training, validation and test curves", all(curves_ok),
      f"{sum(curves_ok)} of {len(RUNS)} runs")

recomputed = grid.loc[grid["val_loss"].idxmin()]
check("after training: the grid choice used validation numbers only",
      (int(recomputed["batch_size"]), float(recomputed["lr"]), int(recomputed["epochs"]))
      == (BEST["batch_size"], BEST["lr"], BEST["epochs"]) and "test" not in inspect.getsource(choose_best).replace("test numbers", ""),
      f"best = lowest val_loss over {len(grid)} combinations; choose_best reads no test column")

## Summary

In [ ]:
checks_table = pd.DataFrame(check_results)
with open(OUT / "model_report.md", "a", encoding="utf-8") as f:
    f.write(f"\n## Checks\n\n{checks_table['passed'].sum()} of {len(checks_table)} passed.\n\n"
            + md_table(checks_table) + "\n")
print(f"Checks passed: {checks_table['passed'].sum()} / {len(checks_table)}")
for row in checks_table[~checks_table["passed"]].itertuples():
    print("  FAILED:", row.check, "-", row.detail)
print()
print(comparison.round(1).to_string(index=False))